# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [13]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item within one order, along with that order's vendor.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [14]:
import pandas as pd

orders_df = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])

assert len(orders_df) == 4
assert orders_df['qty'].sum() == 7

orders_df

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


1st TODO: I called pd.json_normalize with record_path='lines' so each item in an order's lines list becomes its own row, and meta=['order', 'vendor_id'] so every one of those rows keeps its parent order's order and vendor_id. This makes the nested structure into one row per line item.

2nd TODO: I asserted that the table has 4 rows (one per line item across the 3 orders) and that qty sums to 7 (2 + 1 + 1 + 3), which matches the expected values and confirms nothing was dropped or duplicated.

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [15]:
# TODO
vendors_df = pd.DataFrame(vendors_json)

merged = orders_df.merge(vendors_df, on='vendor_id', how='left', indicator=True)

unmatched = merged[merged['_merge'] == 'left_only']
print("Unmatched vendor:", unmatched['vendor_id'].unique())
print("Units attached:", unmatched['qty'].sum())
merged

Unmatched vendor: ['V-07']
Units attached: 3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


I did a left join of the flattened orders to the vendor table on vendor_id with indicator=True, so every order line is kept and the _merge column flags which ones found no vendor. The unmatched vendor is V-07, and it has 3 units attached.

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [16]:
# TODO
# The unmatched vendor has no name (NaN), and groupby silently drops NaN keys by default
# label as "unmatched (v-07) to keep the 3 units visible"
merged['name'] = merged['name'].fillna('UNMATCHED (' + merged['vendor_id'] + ')')

qty_by_vendor = merged.groupby('name')['qty'].sum()
assert qty_by_vendor.sum() == 7
qty_by_vendor

,qty
name,
Cav Merch,1
Hoos Burgers,3
UNMATCHED (V-07),3


I grouped by vendor name and summed qty. Since V-07 has no name, a default groupby would drop it silently. Therefore, I labeled it "UNMATCHED (V-07)" to keep its 3 units visible and make the total add up to 7.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [17]:
# TODO

zone_totals = merged.groupby('zone')['qty'].sum()
print(zone_totals)
print("Zone total:", zone_totals.sum(), "| Overall total:", merged['qty'].sum())
print("Missing units:", merged['qty'].sum() - zone_totals.sum(), "-> belong to unmatched vendor V-07 (no zone)")

zone
A    4
Name: qty, dtype: int64
Zone total: 4 | Overall total: 7
Missing units: 3 -> belong to unmatched vendor V-07 (no zone)


I grouped by zone and summed qty, and zone A sold the most with 4 items. The zone total is 4 instead of 7 because V-07 has no zone, so the 3 units drop out of the groupby. This is where the difference went.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [18]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

with_lines = [o for o in ragged_json if 'lines' in o]
without_lines = [o for o in ragged_json if 'lines' not in o]

flat = pd.json_normalize(with_lines, record_path='lines', meta=['order', 'vendor_id'])
no_lines_df = pd.DataFrame(without_lines)

ragged_df = pd.concat([flat, no_lines_df], ignore_index=True)
ragged_df['qty'] = ragged_df['qty'].astype('Int64')
ragged_df = ragged_df[['order', 'vendor_id', 'item', 'qty']].sort_values('order', ignore_index=True)

print("Rows with missing qty:", ragged_df['qty'].isna().sum())
ragged_df

Rows with missing qty: 2


,order,vendor_id,item,qty
0,4,V-01,Soda,2
1,5,V-10,NaN,<NA>
2,6,V-01,Fries,<NA>


I split out the order with no lines key because json_normalize raises a KeyError on it, flattened the rest, then concatenated that order back in as its own row. I stored qty as null int, so the missing quantity shows as <NA> rather than 0, which keeps "missing" distinguishable from a real zero.

### Q6 — Validate

In [19]:
lines = orders_df
joined = merged

assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) One row per line item is the useful shape here because I only wanted to count and group the items, and each item has its own quantity. With one row per order, the items are stuck inside a list, meaning that asking things like "which item sold the most" would need a lot of digging through the lists. Having one row per line item makes it easier to do Q2, Q3, and Q4 because I just join on vendor and use groupby and sum. It also made the check simple, since 4 rows with quantities adding up to 7 showed nothing got lost.

b) A missing quantity means we don't know how many were sold, while a quantity of zero means we know none were sold. If I filled the missing ones with 0, the totals would look complete when they aren't. Filling the missing quantity with 0 would also hide the problem. Once the Fries line shows a 0, nothing marks it as incomplete, so it would be hard for someone to really notice and fix the source data. If I leave it as missing (`<NA>`), sums still work because they skip it, and I can count how many rows are missing.